# NumPy 파일별 3 · 05_B03 위치 · 구간 고르기와 view · copy

**한 줄 요약** · index · slice로 필요한 부분을 고르고, 고른 결과를 고쳤을 때 원본이 바뀌는 view와 독립된 copy를 구분

- **원본** · `3. numpy/02_단계별실습/05_B03_index_slice_copy_Student.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/03_NumPy` 전체 정리 3-1 ~ 3-5 (위치로 꺼내기 · 뷰와 복사)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 1차원 · 2차원 배열에서 index와 slice로 고른 결과의 값과 shape를 실행 전에 말한다
2. `data[2]` · `data[:, 1]` · `data[1:4, 0:2]`가 시간 · 센서 중 무엇을 고르는지 설명한다
3. slice 결과를 고치면 원본도 바뀌는 이유(view)를 `np.shares_memory()`로 확인한다
4. 원본을 지켜야 할 때 `.copy()`를 붙이고, fancy · mask 선택은 이미 복사본임을 안다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">센서 · 시계열 표에서 특정 시간 구간만 꺼낼 때</td>
<td style="text-align: left;"><code>data[1:4, 0:2]</code> · stop 제외 (1-3)</td>
</tr>
<tr>
<td style="text-align: left;">전처리한 부분 배열을 고쳤는데 원본 데이터가 망가졌을 때</td>
<td style="text-align: left;">view와 <code>.copy()</code> (2-1 · 2-2)</td>
</tr>
<tr>
<td style="text-align: left;">학습 · 검증 데이터를 잘라 따로 다룰 때</td>
<td style="text-align: left;">어떤 선택이 view인지 확인 (2-3)</td>
</tr>
<tr>
<td style="text-align: left;">행 · 열을 바꿔 써서 엉뚱한 센서를 골랐을 때</td>
<td style="text-align: left;"><code>[행, 열]</code> 순서 (1-2)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/nc_map.png" alt="05_B03 학습 지도" width="763">

**예상 학습 시간** · 개념 40분 + 실습 20분

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">실행 환경 · 원본 문제 · 연습 데이터</td>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">위치와 구간 고르기</td>
<td style="text-align: left;">index · slice로 고른 결과의 값과 shape는 어떻게 정해지는가?</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">view와 copy</td>
<td style="text-align: left;">고른 결과를 고치면 원본도 바뀌는가, 어떻게 막는가?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">필요한 구간을 고르고 고쳐도 원본이 그대로이게 하려면?</td>
</tr>
<tr>
<td style="text-align: left;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

원본과 같은 시간 × 센서 연습 데이터를 만듦. **셀은 위에서부터 차례로 실행**.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">설정</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">라이브러리</td>
<td style="text-align: left;"><code>numpy</code></td>
</tr>
<tr>
<td style="text-align: left;">seed · 장치</td>
<td style="text-align: left;">필요 없음 (무작위 · GPU 없음)</td>
</tr>
<tr>
<td style="text-align: left;">데이터</td>
<td style="text-align: left;"><code>sensor</code> (시간 5 × 센서 3) · 원본은 JSON 문자열로 넣었고 여기서는 같은 값을 배열로 직접 씀</td>
</tr>
<tr>
<td style="text-align: left;">축 의미</td>
<td style="text-align: left;">행 = 시간 index · 열 = 센서 index (원본 기준)</td>
</tr>
</tbody>
</table>

In [1]:
# 실행 환경 확인 · 원본과 같은 연습 데이터 준비
import numpy as np  # 배열 계산 라이브러리

print("NumPy:", np.__version__)                     # 설치된 버전 기록
sensor = np.array([[68, 0.31, 101.2],               # 시간 0의 센서 값 3개
                   [71, 0.36, 100.8],
                   [74, 0.44, 102.1],
                   [77, 0.53, 103.4],
                   [79, 0.58, 104.0]])
print("sensor shape:", sensor.shape, sensor.dtype)  # (시간 5, 센서 3) · 실수

NumPy: 2.4.6
sensor shape: (5, 3) float64


### 원본 문제

- **핵심 질문** · 오류 없이 실행돼도 왜 엉뚱한 센서나 시간 구간을 고를 수 있을까?
- **실행 전 예측** · 시간 × 센서 배열에서 `data[2]` · `data[:, 1]` · `data[1:4, 0:2]`의 의미와 shape
- **학생 문제** · 시간 index 1 · 2 · 3 행과 센서 index 0 · 1 열을 고르되, 선택본을 고쳐도 원본이 바뀌면 안 됨
  - 선택식 · shape · copy 필요성을 먼저 제안
  - 출력은 `실제 값 / 결과 모양(shape) / 저장 형식(dtype) / 오류`로 기록
- **다른 조건에서 다시 확인** · 행 = 센서, 열 = 시간으로 축 의미가 바뀐 배열에서 같은 절차 적용

**원본 힌트**

- index는 0부터 세는 위치 번호 · 실제 값과 다름
- slice는 start 포함 · stop 제외
- basic slicing은 view일 수 있음 → 원본 보호가 필요하면 `.copy()`를 명시
- 값이 같아 보이는 것과 메모리를 공유하는 것은 다름 → `np.shares_memory(a, b)`

---
## 1. 위치와 구간 고르기

> **이 장의 흐름**
>
> - 1차원 index → 2차원 `[행, 열]` → 연속 구간 slice 순서로 넓혀 감
> - 고를 때마다 결과 shape를 먼저 예측한 뒤 확인

<img src="images/nc_slice.png" alt="시간 × 센서 표에서 행 하나 · 열 하나 · 직사각형 고르기" width="785">

**파이프라인 위치** · ③ 전처리 — 필요한 시간 구간 · 센서만 꺼내기

### 1-1. `A` index는 위치 번호 · `a[i]` · `a[-1]` <sub>원본 · 먼저 생각</sub>

> 🎯 **`a[1]`의 1은 값인가 위치인가, 범위 밖 번호는 어떻게 되는가?**

💡 **쉽게 말하면**

- index는 **사물함 번호**이고, 값은 사물함 안의 물건.
- 번호는 0부터 세므로 두 번째 사물함이 1번.

**실행 흐름**

1. 값 3개 배열
2. a[1] · a[-1]
3. 범위 밖 a[3] 오류

**먼저 예측** · `a = np.array([10, 20, 30])`에서 `a[1]`은? — 답은 코드 실행 결과 아래에

In [2]:
# 1차원 배열에서 위치 번호로 값 하나 꺼내기
a = np.array([10, 20, 30])                  # 값 3개
print("a[1] =", a[1])                       # 두 번째 위치 = index 1
print("a[-1] =", a[-1])                     # 뒤에서 첫 번째
print("index 1의 값은 20, index 자체는 1")  # 위치 번호와 값은 다름


# 범위 밖 index는 오류
try:
    a[3]                     # 위치는 0 ~ 2뿐
except IndexError as e:      # 오류를 잡아 문장만 출력
    print("IndexError:", e)  # 길이 3 → 마지막 index 2

a[1] = 20
a[-1] = 30
index 1의 값은 20, index 자체는 1
IndexError: index 3 is out of bounds for axis 0 with size 3


**결과** · 예측 정답은 `20` — index 1 = 두 번째 위치

- `a[1] = 20` — 0부터 세므로 두 번째 값
- `a[-1] = 30` — 음수 index는 뒤에서 셈
- `IndexError: index 3 is out of bounds for axis 0 with size 3` — 길이 3 → 마지막 index는 2

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>a[1]</code></td>
<td style="text-align: left;"><strong>위치</strong> 1의 값 하나</td>
<td style="text-align: left;">자리를 알 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>a[-1]</code></td>
<td style="text-align: left;">뒤에서 첫 번째</td>
<td style="text-align: left;">마지막 값 (길이를 몰라도 됨)</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.where(a == 20)[0]</code></td>
<td style="text-align: left;"><strong>값</strong>으로 위치를 찾음</td>
<td style="text-align: left;">값은 알고 자리를 모를 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 사람이 세는 'n번째'는 index n - 1
> - 마지막 값을 `a[len(a)]`로 쓰면 IndexError

> ✅ **한 줄 정리** · index는 0부터 세는 위치 번호이고, 꺼낸 값과는 다르다.

**딥러닝 코드에서는**

- `image, label = dataset[0]` — 첫 번째 샘플
- `losses[-1]` — 마지막 epoch 손실

### 1-2. `A` 2차원 `[행, 열]` · `data[2]` · `data[:, 1]` <sub>원본 · 실행 전 예측</sub>

> 🎯 **시간 × 센서 표에서 `data[2]`와 `data[:, 1]`은 각각 무엇을 고르는가?**

💡 **쉽게 말하면**

- 앞 번호는 **행(시간)**, 뒤 번호는 **열(센서)**.
- `:`는 그 축 전부라는 뜻.

**실행 흐름**

1. data[2] 행 하나
2. data[:, 1] 열 하나
3. data[2, 1] 칸 하나
4. 행 · 열을 바꿔 쓴 경우

**먼저 예측** · `data[2]`와 `data[:, 1]`의 shape는? — 답은 코드 실행 결과 아래에

In [3]:
# 2차원 [행, 열]: 행 = 시간, 열 = 센서
print("data[2] =", sensor[2], sensor[2].shape)           # 시간 index 2의 센서 값 전부
print("data[:, 1] =", sensor[:, 1], sensor[:, 1].shape)  # 센서 1의 모든 시간
print("data[2, 1] =", sensor[2, 1])                      # 시간 2 · 센서 1의 값 하나


# 행 · 열 순서를 바꿔도 오류가 없을 수 있음
print("data[1, 2] =", sensor[1, 2], "| data[2, 1] =", sensor[2, 1])  # 다른 값이 조용히 나옴

data[2] = [ 74.     0.44 102.1 ] (3,)
data[:, 1] = [0.31 0.36 0.44 0.53 0.58] (5,)
data[2, 1] = 0.44
data[1, 2] = 100.8 | data[2, 1] = 0.44


**결과** · 예측 정답은 `(3,)` · `(5,)` — 숫자로 고른 축은 사라짐

- `data[2] = [ 74.     0.44 102.1 ] (3,)` — 시간 2의 센서 3개
- `data[:, 1] = [0.31 0.36 0.44 0.53 0.58] (5,)` — 센서 1의 시간 5개
- `data[2, 1] = 0.44` — 행 2 · 열 1의 값 하나
- `data[1, 2] = 100.8 | data[2, 1] = 0.44` — 순서를 바꾸면 오류 없이 다른 센서 값

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>data[2]</code></td>
<td style="text-align: left;">행 하나 → <code>(센서 수,)</code></td>
<td style="text-align: left;">한 시간의 모든 센서</td>
</tr>
<tr>
<td style="text-align: left;"><code>data[:, 1]</code></td>
<td style="text-align: left;">열 하나 → <code>(시간 수,)</code></td>
<td style="text-align: left;">한 센서의 시간 흐름</td>
</tr>
<tr>
<td style="text-align: left;"><code>data[2, 1]</code></td>
<td style="text-align: left;">칸 하나 → 값</td>
<td style="text-align: left;">특정 시간 · 센서 값</td>
</tr>
<tr>
<td style="text-align: left;"><code>data[2][1]</code></td>
<td style="text-align: left;">두 번 꺼내기 · 같은 값</td>
<td style="text-align: left;">쓰지 않음 (<code>[:][1]</code>은 열이 아니라 행)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 행 · 열을 바꿔 써도 오류가 나지 않음 → shape와 축 의미를 먼저 적기

> ✅ **한 줄 정리** · 2차원 배열은 `[행, 열]` 순서로 고르며, 숫자로 고른 축은 결과에서 사라진다.

**딥러닝 코드에서는**

- `X[:, 0]` — 모든 샘플의 첫 번째 특성
- `series[t]` — 시각 t의 모든 센서 값

### 1-3. `A` slice로 구간 고르기 · `data[1:4, 0:2]` <sub>원본 · 실행 전 예측 · 학생 문제</sub>

> 🎯 **`data[1:4, 0:2]`는 몇 행 몇 열이고, stop은 왜 포함되지 않는가?**

💡 **쉽게 말하면**

- `1:4`는 **1번부터 4번 앞까지** = 1 · 2 · 3.
- 행 구간과 열 구간을 모두 slice로 주면 직사각형이 남음.

**실행 흐름**

1. data[1:4, 0:2]
2. shape (3, 2) 확인
3. stop을 끝값으로 쓴 실수
4. 열 생략

**먼저 예측** · `data[1:4, 0:2]`의 shape는? — 답은 코드 실행 결과 아래에

In [4]:
# slice: start 포함 · stop 제외 → 연속 구간
block = sensor[1:4, 0:2]                     # 시간 1 ~ 3 · 센서 0 ~ 1
print("data[1:4, 0:2] shape:", block.shape)  # (3, 2)
print(block)                                 # 고른 직사각형


# stop을 끝값으로 쓰면 하나 모자람
print("data[1:3] shape:", sensor[1:3].shape)  # 시간 1 · 2 두 줄만
print("data[1:4] == data[1:4, :]:", np.array_equal(sensor[1:4], sensor[1:4, :]))  # 열 생략 = 전체

data[1:4, 0:2] shape: (3, 2)
[[71.    0.36]
 [74.    0.44]
 [77.    0.53]]
data[1:3] shape: (2, 3)
data[1:4] == data[1:4, :]: True


**결과** · 예측 정답은 `(3, 2)` — 행 4 - 1 = 3개, 열 2 - 0 = 2개

- `data[1:4, 0:2] shape: (3, 2)` — 시간 1 · 2 · 3 × 센서 0 · 1
- `data[1:3] shape: (2, 3)` — stop 3 미포함 → 두 줄뿐
- `data[1:4] == data[1:4, :]: True` — 뒤쪽 축을 생략하면 전체

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>data[1:4, 0:2]</code></td>
<td style="text-align: left;">양쪽 slice → <strong>2차원 유지</strong></td>
<td style="text-align: left;">시간 구간 · 센서 묶음</td>
</tr>
<tr>
<td style="text-align: left;"><code>data[1:4, 0]</code></td>
<td style="text-align: left;">열은 숫자 → <code>(3,)</code></td>
<td style="text-align: left;">한 센서의 구간</td>
</tr>
<tr>
<td style="text-align: left;"><code>data[1:4, 0:1]</code></td>
<td style="text-align: left;">열도 slice → <code>(3, 1)</code></td>
<td style="text-align: left;">2차원 모양을 지켜야 할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>data[[1, 2, 3]]</code></td>
<td style="text-align: left;">위치 목록 → 복사본</td>
<td style="text-align: left;">떨어진 위치를 고를 때 (2-3)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - '3번까지'를 원하면 stop은 4
> - slice 결과는 원본과 연결된 view일 수 있음 (2장)

> ✅ **한 줄 정리** · slice는 start 포함 · stop 제외라서 `1:4`는 1 · 2 · 3을 고르며, 양쪽을 slice로 주면 2차원이 남는다.

**딥러닝 코드에서는**

- `X_train, X_val = X[:800], X[800:]`
- `img[:, 10:50, 10:50]` — 이미지 가운데 자르기

---
## 2. view와 copy

> **이 장의 흐름**
>
> - 1장에서 구간을 고르는 법을 봄
> - 고른 결과를 고칠 때 원본이 같이 바뀌는지가 다음 문제 → view · copy · 선택 방법별 차이

<img src="images/nc_view.png" alt="view는 원본 메모리를 같이 쓰고 copy는 새 메모리" width="793">

**파이프라인 위치** · ③ 전처리 — 원본 데이터 보호

### 2-1. `A` view를 고치면 원본도 바뀜 · `np.shares_memory()` <sub>원본 코드 셀 앞부분</sub>

> 🎯 **slice로 고른 `view`의 첫 칸을 고치면 원본 `sensor`는 어떻게 되는가?**

💡 **쉽게 말하면**

- slice 결과는 원본을 들여다보는 **창문(view)**.
- 창문 너머 물건을 옮기면 원본 방의 물건이 옮겨짐.

**실행 흐름**

1. 원본 보관
2. view = sensor[1:3, :2]
3. view[0, 0] = -999
4. 원본 · 공유 확인
5. 되돌리기

**먼저 예측** · `view[0, 0] = -999` 뒤 `sensor[1, 0]`은? — 답은 코드 실행 결과 아래에

In [5]:
# 원본 코드: slice로 고른 결과(view)를 고침
original = sensor.copy()                                 # 실험 뒤 되돌릴 원본 보관
view = sensor[1:3, :2]                                   # 시간 1 ~ 2 · 센서 0 ~ 1
view[0, 0] = -999                                        # 고른 쪽의 첫 칸만 수정
print("view 수정이 원본에 반영:", sensor[1, 0] == -999)  # 원본도 바뀌었나
print("메모리 공유:", np.shares_memory(sensor, view))    # 같은 메모리를 보는가


# 원본 되돌리기
sensor = original                           # 보관해 둔 값으로 교체
print("되돌린 원본 [1, 0]:", sensor[1, 0])  # 71.0

view 수정이 원본에 반영: True
메모리 공유: True
되돌린 원본 [1, 0]: 71.0


**결과** · 예측 정답은 `-999` — view와 원본은 같은 메모리

- `view 수정이 원본에 반영: True` — view의 [0, 0] = 원본의 [1, 0]
- `메모리 공유: True` — 같은 메모리를 봄
- `되돌린 원본 [1, 0]: 71.0` — 보관해 둔 사본으로 복구

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>sensor[1:3, :2]</code></td>
<td style="text-align: left;"><strong>view</strong> · 원본과 연결</td>
<td style="text-align: left;">읽기만 할 때 (빠름 · 메모리 절약)</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.shares_memory(a, b)</code></td>
<td style="text-align: left;">같은 메모리인지 <strong>True/False</strong></td>
<td style="text-align: left;">원본 연결 여부를 확인할 때</td>
</tr>
<tr>
<td style="text-align: left;">값 비교 <code>np.array_equal</code></td>
<td style="text-align: left;">값만 비교 · 연결 여부는 모름</td>
<td style="text-align: left;">결과가 같은지 볼 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - Python 리스트 슬라이싱은 복사라 습관이 반대
> - view를 고치기 전 원본이 필요한지 먼저 판단

> ✅ **한 줄 정리** · NumPy slice 결과는 원본 메모리를 같이 쓰는 view라서, 고치면 원본도 바뀐다.

**딥러닝 코드에서는**

- `batch = X[i:i + 32]` — 읽기만 하면 view라 빠름
- 증강처럼 값을 바꿀 땐 복사본에서

### 2-2. `A` `.copy()`로 원본 보호 <sub>원본 코드 셀 뒷부분</sub>

> 🎯 **같은 구간을 `.copy()`로 고르면 원본은 어떻게 되는가?**

💡 **쉽게 말하면**

- `.copy()`는 창문 대신 **사진을 찍어 따로 가져가는 것**.
- 사진에 낙서해도 원본은 그대로.

**실행 흐름**

1. safe = sensor[1:3, :2].copy()
2. safe[0, 0] = -999
3. 원본 · 공유 확인

**먼저 예측** · `.copy()` 뒤 `safe[0, 0] = -999`를 하면 원본 `sensor[1, 0]`은? — 답은 코드 실행 결과 아래에

In [6]:
# 원본 코드: .copy()로 고른 결과를 따로 만든 뒤 고침
safe = sensor[1:3, :2].copy()                           # 같은 구간 · 새 메모리
safe[0, 0] = -999                                       # 복사본만 수정
print("copy 수정 뒤 원본 보존:", sensor[1, 0] != -999)  # 원본 그대로인가
print("메모리 공유:", np.shares_memory(sensor, safe))   # 다른 메모리
print("safe:", safe.tolist())                           # 복사본 값

copy 수정 뒤 원본 보존: True
메모리 공유: False
safe: [[-999.0, 0.36], [74.0, 0.44]]


**결과** · 예측 정답은 `71.0` 그대로 — 새 메모리

- `copy 수정 뒤 원본 보존: True` — 원본 [1, 0]은 71.0 그대로
- `메모리 공유: False` — 다른 메모리
- `safe: [[-999.0, 0.36], [74.0, 0.44]]` — 복사본만 바뀜

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>a[1:3].copy()</code></td>
<td style="text-align: left;"><strong>독립</strong> 복사본</td>
<td style="text-align: left;">고칠 예정일 때 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>a[1:3]</code></td>
<td style="text-align: left;">view · 원본과 연결</td>
<td style="text-align: left;">읽기만 할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.array(a[1:3])</code></td>
<td style="text-align: left;">기본적으로 복사</td>
<td style="text-align: left;">다른 자료형에서 배열을 만들 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 큰 배열을 매번 복사하면 메모리 사용이 늘어남 → 고칠 때만 복사

> ✅ **한 줄 정리** · 고칠 예정인 선택 결과는 `.copy()`로 새 메모리를 만들어 원본을 지킨다.

**딥러닝 코드에서는**

- `x_aug = x.copy()` 후 증강
- PyTorch는 `tensor.clone()`

### 2-3. `B` 어떤 선택이 view인가 · `a[[1, 2]]` · `a[mask]` · `reshape()` <sub>보충 · 원본 개념 확장</sub>

> 🎯 **slice · fancy · mask · reshape 중 원본과 메모리를 공유하는 것은?**

💡 **쉽게 말하면**

- **규칙적인 구간**(slice) · **모양만 바꾸기**(reshape)는 원본을 그대로 보여 줌.
- **위치 목록 · 조건**으로 고르면 결과를 새로 모아 만듦.

**실행 흐름**

1. 네 가지 방법으로 고르기
2. 방법마다 shares_memory

**먼저 예측** · `sensor[[1, 2]]`는 원본과 메모리를 공유할까? — 답은 코드 실행 결과 아래에

In [7]:
# 어떤 선택이 view이고 어떤 선택이 copy인가
picks = {"slice  sensor[1:3]": sensor[1:3],                              # 연속 구간
         "fancy  sensor[[1, 2]]": sensor[[1, 2]],                        # 위치 목록
         "mask   sensor[sensor[:, 0] > 72]": sensor[sensor[:, 0] > 72],  # 조건
         "reshape sensor.reshape(3, 5)": sensor.reshape(3, 5)}           # 모양만 바꿈
for name, arr in picks.items():                                          # 방법마다
    print(f"{name:<34} 공유: {np.shares_memory(sensor, arr)}")           # 원본 메모리를 같이 쓰나

slice  sensor[1:3]                 공유: True
fancy  sensor[[1, 2]]              공유: False
mask   sensor[sensor[:, 0] > 72]   공유: False
reshape sensor.reshape(3, 5)       공유: True


**결과** · 예측 정답은 공유하지 않음 — 위치 목록(fancy) 선택은 복사본

- `slice  sensor[1:3]                 공유: True` — slice는 view
- `fancy  sensor[[1, 2]]              공유: False` — 위치 목록은 복사
- `mask   sensor[sensor[:, 0] > 72]   공유: False` — 조건 선택도 복사
- `reshape sensor.reshape(3, 5)       공유: True` — reshape는 가능하면 view

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">slice <code>a[1:3]</code></td>
<td style="text-align: left;">view</td>
<td style="text-align: left;">구간 읽기</td>
</tr>
<tr>
<td style="text-align: left;">fancy <code>a[[1, 2]]</code></td>
<td style="text-align: left;">복사</td>
<td style="text-align: left;">떨어진 위치 고르기</td>
</tr>
<tr>
<td style="text-align: left;">mask <code>a[a &gt; 0]</code></td>
<td style="text-align: left;">복사</td>
<td style="text-align: left;">조건으로 고르기</td>
</tr>
<tr>
<td style="text-align: left;"><code>reshape()</code></td>
<td style="text-align: left;">가능하면 view</td>
<td style="text-align: left;">모양만 바꿀 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - view인지 외우기보다 `np.shares_memory()`로 확인
> - reshape도 원본과 연결될 수 있음

> ✅ **한 줄 정리** · slice · reshape는 원본과 연결되고 fancy · mask 선택은 복사본이므로, 헷갈리면 `np.shares_memory()`로 확인한다.

**딥러닝 코드에서는**

- `X[train_idx]` — 섞은 번호로 고르면 복사본이 만들어짐
- `x.view(...)` · `x.reshape(...)` — PyTorch도 모양 바꾸기는 메모리 공유

---
## 3. 종합 실습 · 원본을 지키며 구간 고르기

> **이 장의 흐름**
>
> - 1 · 2장을 원본 학생 문제에 그대로 적용
> - 축 의미가 바뀐 배열에서 같은 절차를 다시 확인

<img src="images/nc_practice.png" alt="시간 1 ~ 3 · 센서 0 ~ 1을 복사해 고르기" width="715">

**파이프라인 위치** · ① 데이터 준비 → ③ 전처리

> 🎯 **시간 index 1 ~ 3 · 센서 index 0 ~ 1을 원본과 독립적으로 고르려면?**

💡 **쉽게 말하면**

- **축 의미 → 범위 → 예상 shape → 원본 변경 위험** 순서로 먼저 적고 코드를 씀.
- 고칠 예정이므로 `.copy()`를 붙임.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 문법</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">축 의미 적기: 행 = 시간 · 열 = 센서</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">범위: 행 <code>1:4</code> · 열 <code>0:2</code></td>
<td style="text-align: left;">1-3</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">예상 shape <code>(3, 2)</code></td>
<td style="text-align: left;">1-3</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;"><code>.copy()</code>로 독립 선택</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">선택본을 고친 뒤 원본 · 공유 확인</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">축 의미가 바뀐 배열에서 다시 확인</td>
<td style="text-align: left;">1-2 · 2-2</td>
</tr>
</tbody>
</table>

**먼저 예측** · 선택본 `[0, 0]`을 -1로 고친 뒤 원본 `problem_data[1, 0]`은? — 답은 코드 실행 결과 아래에

In [8]:
# 원본 학생 문제: 시간 행 1 ~ 3 · 센서 열 0 ~ 1을 원본과 독립적으로 고르기
problem_data = np.arange(24).reshape(6, 4)  # 시간 6 × 센서 4
predicted_shape = (3, 2)                    # 코드 전에 적은 예상 shape
selection = problem_data[1:4, 0:2].copy()   # 범위 선택 + 복사
# 비교
print("원본 shape:", problem_data.shape, "| 예상:", predicted_shape, "| 실제:", selection.shape)
print(selection)                            # 고른 값


# 선택본을 고쳐도 원본이 그대로인지 확인
selection[0, 0] = -1  # 선택본만 수정
# 4 · False
print("원본 [1, 0]:", problem_data[1, 0], "| 공유:", np.shares_memory(problem_data, selection))

원본 shape: (6, 4) | 예상: (3, 2) | 실제: (3, 2)
[[ 4  5]
 [ 8  9]
 [12 13]]
원본 [1, 0]: 4 | 공유: False


In [9]:
# 원본 '다른 조건에서 다시 확인': 이번에는 행 = 센서, 열 = 시간
retry_data = np.arange(15).reshape(3, 5)  # 센서 3 × 시간 5
retry_copy = retry_data[:, 1:4].copy()    # 시간 1 ~ 3을 고름 (축 의미가 바뀜)
retry_copy[0, 0] = -1                     # 복사본만 수정
assert retry_data[0, 1] != -1             # 원본은 그대로여야 함
print("축 의미 변경 뒤 원본 불변:", retry_data.shape, retry_copy.shape)  # (3, 5) (3, 3)

축 의미 변경 뒤 원본 불변: (3, 5) (3, 3)


**결과** · 예측 정답은 `4` 그대로 — 복사본이므로 원본과 연결되지 않음

- `원본 shape: (6, 4) | 예상: (3, 2) | 실제: (3, 2)` — 예상한 shape와 같음
- 고른 값은 `[[4, 5], [8, 9], [12, 13]]` (시간 1 ~ 3 · 센서 0 ~ 1)
- `원본 [1, 0]: 4 | 공유: False` — 선택본을 고쳐도 원본 그대로
- `축 의미 변경 뒤 원본 불변: (3, 5) (3, 3)` — 행 = 센서일 때는 시간 구간이 열 slice가 됨

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>problem_data[1:4, 0:2].copy()</code></td>
<td style="text-align: left;">구간 + 독립</td>
<td style="text-align: left;">고칠 예정일 때 (<strong>기본 선택</strong>)</td>
</tr>
<tr>
<td style="text-align: left;"><code>problem_data[1:4, 0:2]</code></td>
<td style="text-align: left;">구간 · view</td>
<td style="text-align: left;">읽기만 할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>problem_data[[1, 2, 3]][:, [0, 1]]</code></td>
<td style="text-align: left;">fancy 두 번 · 복사</td>
<td style="text-align: left;">떨어진 위치일 때만</td>
</tr>
<tr>
<td style="text-align: left;"><code>retry_data[:, 1:4]</code></td>
<td style="text-align: left;">축 의미가 바뀌면 slice 위치도 바뀜</td>
<td style="text-align: left;">행 = 센서인 배열</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 축 의미가 바뀌면 같은 '시간 1 ~ 3'도 행이 아니라 열에서 고름
> - shape만 맞아도 엉뚱한 축을 골랐을 수 있음 → 축 의미부터 적기

> ✅ **한 줄 정리** · 축 의미와 범위를 먼저 적고 예상 shape와 비교한 뒤, 고칠 선택은 `.copy()`로 원본과 끊는다.

**딥러닝 코드에서는**

- `X_val = X[800:].copy()` — 검증 데이터를 따로 정규화할 때
- 시계열 `(시간, 센서)`와 `(배치, 시간, 특성)`처럼 축 순서를 코드 위에 주석으로

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. index · slice 결과 예측</td>
<td style="text-align: left;"><code>a[1]</code> = 20, <code>data[1:4, 0:2]</code> → <code>(3, 2)</code></td>
<td style="text-align: left;">1-1 · 1-3</td>
</tr>
<tr>
<td style="text-align: left;">2. 축 의미 읽기</td>
<td style="text-align: left;"><code>data[2]</code>는 시간 2의 센서 3개, <code>data[:, 1]</code>은 센서 1의 시간 5개</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">3. view 확인</td>
<td style="text-align: left;">view를 고치면 원본 [1, 0]이 -999, <code>shares_memory</code> True</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">4. copy · 선택 방법별 차이</td>
<td style="text-align: left;"><code>.copy()</code>는 공유 False, fancy · mask도 복사, slice · reshape는 공유</td>
<td style="text-align: left;">2-2 · 2-3 · 3</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">index</td>
<td style="text-align: left;">0부터 세는 위치 번호</td>
<td style="text-align: left;"><code>dataset[0]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;"><code>[행, 열]</code></td>
<td style="text-align: left;">숫자로 고른 축은 사라짐</td>
<td style="text-align: left;"><code>X[:, 0]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">slice</td>
<td style="text-align: left;">start 포함 · stop 제외</td>
<td style="text-align: left;"><code>X[:800]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">view</td>
<td style="text-align: left;">slice 결과를 고치면 원본도</td>
<td style="text-align: left;"><code>batch = X[i:i + 32]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;"><code>.copy()</code></td>
<td style="text-align: left;">고칠 선택은 복사</td>
<td style="text-align: left;"><code>x.copy()</code> · <code>tensor.clone()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">fancy · mask</td>
<td style="text-align: left;">이미 복사본</td>
<td style="text-align: left;"><code>X[train_idx]</code></td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;"><code>a[1]</code>의 1은 값인가 위치인가?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;"><code>data[2]</code>와 <code>data[:, 1]</code>은 무엇을 고르는가?</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;"><code>data[1:4, 0:2]</code>의 shape는?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">slice 결과를 고치면 원본은?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">원본을 지키며 고르려면?</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">fancy · mask 선택은 view인가?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `m = np.arange(20).reshape(4, 5)`에서 행 1 ~ 2 · 열 2 ~ 4를 **복사**로 골라 shape 출력
2. 그 선택본의 `[0, 0]`을 0으로 바꾼 뒤 원본 `m[1, 2]`와 메모리 공유 여부 출력

In [10]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. `sensor[1:4, ::2]`의 shape와 고른 센서 번호를 예측 (step 2)
2. `v = sensor[:, 0]` 뒤 `v += 1`을 하면 원본 첫 열은? `v = sensor[:, 0] + 1`이면?
3. `sensor[sensor[:, 0] > 72][0, 0] = 0`이 원본을 바꾸지 못하는 이유 말하기

In [11]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">위치(index) · 0부터 세므로 두 번째 값</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">시간 2의 모든 센서(행) · 센서 1의 모든 시간(열)</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;"><code>(3, 2)</code> — stop 제외</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">같이 바뀜(view) · <code>np.shares_memory</code>가 True</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;"><code>.copy()</code>를 붙여 새 메모리로</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">아님 · 복사본 (slice · reshape는 view)</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
m = np.arange(20).reshape(4, 5)
part = m[1:3, 2:5].copy()
print(part.shape)  # (2, 3)
part[0, 0] = 0
print(m[1, 2], np.shares_memory(m, part))  # 7 False
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><code>(3, 2)</code> — 행 1 · 2 · 3, 센서 0 · 2 (step 2로 건너뜀)</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><code>v += 1</code>은 view를 제자리에서 고쳐 원본 첫 열도 1씩 커짐 · <code>+ 1</code>은 새 배열이라 원본 그대로</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">mask 선택이 만든 <strong>복사본</strong>을 고친 것이라 원본과 연결되지 않음</td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) <code>data[1:3]</code>이 3줄이라고 답함</td>
<td style="text-align: left;">stop을 포함한다고 착각</td>
<td style="text-align: left;">stop은 제외 → 1 · 2 두 줄</td>
<td style="text-align: left;">stop 번호 앞에서 멈췄나?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. NumPy에서 view와 copy의 차이는 무엇인가요?**
> - view는 원본과 같은 메모리를 보는 선택 결과라 값을 고치면 원본도 바뀌고, copy는 새 메모리에 만든 독립 배열입니다.
> - 기본 slice와 reshape는 view, 위치 목록이나 조건으로 고른 결과는 copy이며, 헷갈리면 np.shares_memory로 확인합니다.

**Q2. 2차원 배열에서 행과 열을 바꿔 써도 오류가 나지 않는 이유와 대처는?**
> - 두 번호가 각 축 길이 안에만 있으면 문법상 문제가 없어서 엉뚱한 값이 조용히 나옵니다.
> - 그래서 코드 전에 축 의미(행은 시간, 열은 센서)와 예상 shape를 적고, 결과 shape를 출력해 맞는지 확인합니다.

**Q3. 전처리에서 원본 데이터를 보호하려면 어떻게 하나요?**
> - 값을 바꿀 부분 배열은 .copy()로 새 메모리에 만들고, 원본 배열은 다시 쓰지 않는 변수로 남겨 둡니다.
> - 읽기만 할 때는 view를 써서 메모리를 아낍니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>a[i]</code> · <code>a[-1]</code></td>
<td style="text-align: left;">NumPy 인덱싱</td>
<td style="text-align: left;">위치로 값 하나</td>
<td style="text-align: left;">1-1 <code>a[1]</code> · <code>a[-1]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>data[행, 열]</code> · <code>data[:, j]</code></td>
<td style="text-align: left;">NumPy 인덱싱</td>
<td style="text-align: left;">칸 · 행 · 열 고르기</td>
<td style="text-align: left;">1-2 <code>sensor[2, 1]</code> · <code>sensor[:, 1]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>data[a:b, c:d]</code></td>
<td style="text-align: left;">NumPy 인덱싱 (slice)</td>
<td style="text-align: left;">직사각형 구간 (view)</td>
<td style="text-align: left;">1-3 <code>sensor[1:4, 0:2]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.copy()</code></td>
<td style="text-align: left;">ndarray 메서드</td>
<td style="text-align: left;">독립 복사본</td>
<td style="text-align: left;">2-2 <code>sensor[1:3, :2].copy()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.shares_memory()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">메모리 공유 확인</td>
<td style="text-align: left;">2-1 <code>np.shares_memory(sensor, view)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.array_equal()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">값이 모두 같은지</td>
<td style="text-align: left;">1-3 <code>np.array_equal(sensor[1:4], sensor[1:4, :])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>a[[i, j]]</code> · <code>a[mask]</code></td>
<td style="text-align: left;">NumPy 인덱싱 (fancy · mask)</td>
<td style="text-align: left;">복사본으로 고르기</td>
<td style="text-align: left;">2-3 <code>sensor[[1, 2]]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.arange().reshape()</code></td>
<td style="text-align: left;">numpy · ndarray 메서드</td>
<td style="text-align: left;">연습 배열 만들기</td>
<td style="text-align: left;">3 <code>np.arange(24).reshape(6, 4)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>assert</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">조건이 거짓이면 멈춤</td>
<td style="text-align: left;">3 <code>assert retry_data[0, 1] != -1</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">축 의미 · 예상 shape를 코드 위에 적기</td>
<td style="text-align: left;">행 · 열 순서를 감으로 쓰기</td>
</tr>
<tr>
<td style="text-align: left;">고칠 선택은 <code>.copy()</code></td>
<td style="text-align: left;">slice 결과를 원본 걱정 없이 수정</td>
</tr>
<tr>
<td style="text-align: left;">헷갈리면 <code>np.shares_memory()</code></td>
<td style="text-align: left;">값이 같으니 복사라고 판단</td>
</tr>
<tr>
<td style="text-align: left;">'n까지'는 stop = n + 1</td>
<td style="text-align: left;">stop에 끝 번호를 그대로</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">index vs 값</td>
<td style="text-align: left;">위치 번호 vs 그 자리의 내용</td>
<td style="text-align: left;"><code>a[1]</code>의 1은 위치</td>
</tr>
<tr>
<td style="text-align: left;"><code>data[2]</code> vs <code>data[:, 2]</code></td>
<td style="text-align: left;">행 하나 vs 열 하나</td>
<td style="text-align: left;">앞 = 행</td>
</tr>
<tr>
<td style="text-align: left;"><code>data[1:4, 0]</code> vs <code>data[1:4, 0:1]</code></td>
<td style="text-align: left;"><code>(3,)</code> vs <code>(3, 1)</code></td>
<td style="text-align: left;">모양 유지는 slice</td>
</tr>
<tr>
<td style="text-align: left;">view vs copy</td>
<td style="text-align: left;">원본과 연결 vs 독립</td>
<td style="text-align: left;">고칠 땐 copy</td>
</tr>
<tr>
<td style="text-align: left;">slice vs fancy</td>
<td style="text-align: left;">view vs 복사</td>
<td style="text-align: left;">헷갈리면 <code>shares_memory</code></td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">10분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개 다시 풀기 · 그림을 가리고 shape 말하기</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">3장 종합 실습을 빈 셀에 직접 다시 짜기</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 · 전체 정리 3장 ✅ 줄만 읽기</td>
<td style="text-align: left;">10분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 · `05_B03_index_slice_copy_Student.ipynb`의 용어 설명 · 예측 · 코드 셀 · 학생 문제 · 다시 확인
- 1장 · 용어 설명과 예측 질문을 실행 코드로 · 2장 · 원본 코드 셀을 둘로 나누고 선택 방법별 차이를 보충
- 원본 JSON 문자열 데이터는 같은 값의 배열로 · 오류 예시 스위치는 try/except로 바꿈
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">먼저 알아둘 배열 말 · 먼저 생각</td>
<td style="text-align: left;">index · slice · view · copy 용어</td>
<td style="text-align: left;">1-1 · 2장</td>
</tr>
<tr>
<td style="text-align: left;">실행 전 예측</td>
<td style="text-align: left;"><code>data[2]</code> · <code>data[:, 1]</code> · <code>data[1:4, 0:2]</code></td>
<td style="text-align: left;">1-2 · 1-3</td>
</tr>
<tr>
<td style="text-align: left;">개념 · 개념 도식</td>
<td style="text-align: left;">start 포함 · stop 제외 · view · copy</td>
<td style="text-align: left;">1-3 · 2장 그림</td>
</tr>
<tr>
<td style="text-align: left;">코드 셀 (view · copy)</td>
<td style="text-align: left;">view 수정 → 원본 반영 · copy 보존</td>
<td style="text-align: left;">2-1 · 2-2</td>
</tr>
<tr>
<td style="text-align: left;">학생 문제 (TODO)</td>
<td style="text-align: left;">시간 1 ~ 3 · 센서 0 ~ 1 독립 선택</td>
<td style="text-align: left;">3 종합 실습</td>
</tr>
<tr>
<td style="text-align: left;">다른 조건에서 교정 · 재시도</td>
<td style="text-align: left;">행 = 센서로 축 의미가 바뀐 배열</td>
<td style="text-align: left;">3 종합 실습</td>
</tr>
<tr>
<td style="text-align: left;">응용 · 자기 점검</td>
<td style="text-align: left;">축 의미 → 범위 → shape → 원본 변경 위험</td>
<td style="text-align: left;">3 쉽게 말하면 · 오답 노트</td>
</tr>
</tbody>
</table>

### 전체 코드

1 ~ 3장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

In [12]:
# ---------- 1-1 ----------
# 1차원 배열에서 위치 번호로 값 하나 꺼내기
a = np.array([10, 20, 30])                  # 값 3개
print("a[1] =", a[1])                       # 두 번째 위치 = index 1
print("a[-1] =", a[-1])                     # 뒤에서 첫 번째
print("index 1의 값은 20, index 자체는 1")  # 위치 번호와 값은 다름


# 범위 밖 index는 오류
try:
    a[3]                     # 위치는 0 ~ 2뿐
except IndexError as e:      # 오류를 잡아 문장만 출력
    print("IndexError:", e)  # 길이 3 → 마지막 index 2


# ---------- 1-2 ----------
# 2차원 [행, 열]: 행 = 시간, 열 = 센서
print("data[2] =", sensor[2], sensor[2].shape)           # 시간 index 2의 센서 값 전부
print("data[:, 1] =", sensor[:, 1], sensor[:, 1].shape)  # 센서 1의 모든 시간
print("data[2, 1] =", sensor[2, 1])                      # 시간 2 · 센서 1의 값 하나


# 행 · 열 순서를 바꿔도 오류가 없을 수 있음
print("data[1, 2] =", sensor[1, 2], "| data[2, 1] =", sensor[2, 1])  # 다른 값이 조용히 나옴


# ---------- 1-3 ----------
# slice: start 포함 · stop 제외 → 연속 구간
block = sensor[1:4, 0:2]                     # 시간 1 ~ 3 · 센서 0 ~ 1
print("data[1:4, 0:2] shape:", block.shape)  # (3, 2)
print(block)                                 # 고른 직사각형


# stop을 끝값으로 쓰면 하나 모자람
print("data[1:3] shape:", sensor[1:3].shape)  # 시간 1 · 2 두 줄만
print("data[1:4] == data[1:4, :]:", np.array_equal(sensor[1:4], sensor[1:4, :]))  # 열 생략 = 전체

a[1] = 20
a[-1] = 30
index 1의 값은 20, index 자체는 1
IndexError: index 3 is out of bounds for axis 0 with size 3
data[2] = [ 74.     0.44 102.1 ] (3,)
data[:, 1] = [0.31 0.36 0.44 0.53 0.58] (5,)
data[2, 1] = 0.44
data[1, 2] = 100.8 | data[2, 1] = 0.44
data[1:4, 0:2] shape: (3, 2)
[[71.    0.36]
 [74.    0.44]
 [77.    0.53]]
data[1:3] shape: (2, 3)
data[1:4] == data[1:4, :]: True


In [13]:
# ---------- 2-1 ----------
# 원본 코드: slice로 고른 결과(view)를 고침
original = sensor.copy()                                 # 실험 뒤 되돌릴 원본 보관
view = sensor[1:3, :2]                                   # 시간 1 ~ 2 · 센서 0 ~ 1
view[0, 0] = -999                                        # 고른 쪽의 첫 칸만 수정
print("view 수정이 원본에 반영:", sensor[1, 0] == -999)  # 원본도 바뀌었나
print("메모리 공유:", np.shares_memory(sensor, view))    # 같은 메모리를 보는가


# 원본 되돌리기
sensor = original                           # 보관해 둔 값으로 교체
print("되돌린 원본 [1, 0]:", sensor[1, 0])  # 71.0


# ---------- 2-2 ----------
# 원본 코드: .copy()로 고른 결과를 따로 만든 뒤 고침
safe = sensor[1:3, :2].copy()                           # 같은 구간 · 새 메모리
safe[0, 0] = -999                                       # 복사본만 수정
print("copy 수정 뒤 원본 보존:", sensor[1, 0] != -999)  # 원본 그대로인가
print("메모리 공유:", np.shares_memory(sensor, safe))   # 다른 메모리
print("safe:", safe.tolist())                           # 복사본 값


# ---------- 2-3 ----------
# 어떤 선택이 view이고 어떤 선택이 copy인가
picks = {"slice  sensor[1:3]": sensor[1:3],                              # 연속 구간
         "fancy  sensor[[1, 2]]": sensor[[1, 2]],                        # 위치 목록
         "mask   sensor[sensor[:, 0] > 72]": sensor[sensor[:, 0] > 72],  # 조건
         "reshape sensor.reshape(3, 5)": sensor.reshape(3, 5)}           # 모양만 바꿈
for name, arr in picks.items():                                          # 방법마다
    print(f"{name:<34} 공유: {np.shares_memory(sensor, arr)}")           # 원본 메모리를 같이 쓰나

view 수정이 원본에 반영: True
메모리 공유: True
되돌린 원본 [1, 0]: 71.0
copy 수정 뒤 원본 보존: True
메모리 공유: False
safe: [[-999.0, 0.36], [74.0, 0.44]]
slice  sensor[1:3]                 공유: True
fancy  sensor[[1, 2]]              공유: False
mask   sensor[sensor[:, 0] > 72]   공유: False
reshape sensor.reshape(3, 5)       공유: True


In [14]:
# ---------- 3. 종합 실습 ----------
# 원본 학생 문제: 시간 행 1 ~ 3 · 센서 열 0 ~ 1을 원본과 독립적으로 고르기
problem_data = np.arange(24).reshape(6, 4)  # 시간 6 × 센서 4
predicted_shape = (3, 2)                    # 코드 전에 적은 예상 shape
selection = problem_data[1:4, 0:2].copy()   # 범위 선택 + 복사
# 비교
print("원본 shape:", problem_data.shape, "| 예상:", predicted_shape, "| 실제:", selection.shape)
print(selection)                            # 고른 값


# 선택본을 고쳐도 원본이 그대로인지 확인
selection[0, 0] = -1  # 선택본만 수정
# 4 · False
print("원본 [1, 0]:", problem_data[1, 0], "| 공유:", np.shares_memory(problem_data, selection))


# 원본 '다른 조건에서 다시 확인': 이번에는 행 = 센서, 열 = 시간
retry_data = np.arange(15).reshape(3, 5)  # 센서 3 × 시간 5
retry_copy = retry_data[:, 1:4].copy()    # 시간 1 ~ 3을 고름 (축 의미가 바뀜)
retry_copy[0, 0] = -1                     # 복사본만 수정
assert retry_data[0, 1] != -1             # 원본은 그대로여야 함
print("축 의미 변경 뒤 원본 불변:", retry_data.shape, retry_copy.shape)  # (3, 5) (3, 3)

원본 shape: (6, 4) | 예상: (3, 2) | 실제: (3, 2)
[[ 4  5]
 [ 8  9]
 [12 13]]
원본 [1, 0]: 4 | 공유: False
축 의미 변경 뒤 원본 불변: (3, 5) (3, 3)
